# EEG_16 — Clustering Soggetti per Matrice di Connettività

**Direzione**: Francesco Pelosin — approccio data-first (non accuracy-first)

**Logica**:
1. Calcola matrice di connettività media per soggetto (5 metriche: pcc, abs_pcc, im_pcc, wpli, plv)
2. Clustering soggetti con K-means (k=2,3,4) → silhouette per scegliere k ottimale
3. Matrice media per cluster → confronto visivo inter-cluster
4. Post-hoc: overlay accuracy EEG_12/13 → i cluster EEG predicono la performance?

**Riferimento**: EEG_08b mostra variabilità inter-soggetto dominante (ε²=0.85). Questo notebook trova la struttura.

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import torch
from scipy.signal import hilbert
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg16')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg16'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_CHANNELS     = 61
N_SAMPLES      = 384
N_CLASSES      = 4
CLUSTER_SCHEME = 'concr4'

METRICS        = ['pcc', 'abs_pcc', 'im_pcc', 'wpli', 'plv']
K_LIST         = [2, 3, 4]       # numero cluster da esplorare
N_INIT_KMEANS  = 20              # riavvii K-means per stabilità
RANDOM_SEED    = 42

# Carica accuracy EEG_12/13 se disponibile (post-hoc)
ACC_FILE_12 = project_root / 'figures' / 'eeg12_subject_ranking.csv'
ACC_FILE_13 = project_root / 'figures' / 'eeg13b_subject_ranking.csv'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()).items()}

# Indice dei trial per metrica → {sid: [path, ...]}
def build_index(metric):
    root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
    idx = defaultdict(list)
    if not root.exists():
        log.warning(f'Directory non trovata: {root}')
        return idx
    for p in sorted(root.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if m:
            idx[int(m.group(1))].append(p)
    return idx

# Usa abs_pcc come metrica principale per la lista soggetti
MAIN_IDX = build_index('abs_pcc')
ALL_SUBJ = sorted(MAIN_IDX.keys())
log.info(f'Soggetti trovati: {len(ALL_SUBJ)}')
log.info(f'Metriche: {METRICS}')

## §2 — Caricamento Matrici di Connettività Per-Soggetto

Legge `d['adj']` dai file `data/hypergraphs_pruned_{metric}/` già generati da **EEG_07f**.
Nessun ricalcolo — media delle matrici trial-level → `(61, 61)` fingerprint per soggetto.

Alla prima esecuzione salva `models/eeg16/subject_conn_matrices.npz` (cache).
Dalla seconda esecuzione in poi: caricamento istantaneo dalla cache.

In [ ]:
# ── Cache per-soggetto ────────────────────────────────────────────────────────
CONN_CACHE = CKPT_DIR / 'subject_conn_matrices.npz'

if CONN_CACHE.exists():
    log.info(f'Cache trovata: {CONN_CACHE}')
    _c = np.load(CONN_CACHE, allow_pickle=True)
    CONN     = _c['conn'].item()        # dict: metric → (n_subj, 61, 61)
    SUBJ_IDS = _c['subj_ids'].tolist()
else:
    log.info('Calcolo matrici di connettività per-soggetto da .pt (campo adj)...')
    CONN     = {m: [] for m in METRICS}
    SUBJ_IDS = []

    for sid in tqdm(ALL_SUBJ, desc='Soggetti'):
        # Per ogni metrica, usa la directory specifica — build_index per quella metrica
        # Prima verifica che il soggetto abbia dati su almeno la metrica principale
        paths_main = MAIN_IDX[sid]
        if len(paths_main) < 10:
            continue

        accum = {m: [] for m in METRICS}

        for metric in METRICS:
            idx_m = build_index(metric)
            paths_m = idx_m.get(sid, [])
            for p in paths_m:
                try:
                    d = torch.load(p, weights_only=False)
                    adj = d['adj'].float().numpy()   # (61, 61) già calcolata da EEG_07f
                    if adj.shape == (N_CHANNELS, N_CHANNELS):
                        accum[metric].append(adj)
                except Exception:
                    continue

        # Soggetto valido se ha almeno 5 trial su ogni metrica
        if any(len(accum[m]) < 5 for m in METRICS):
            continue

        for m in METRICS:
            CONN[m].append(np.mean(accum[m], axis=0))   # (61, 61) media trial
        SUBJ_IDS.append(sid)

    for m in METRICS:
        CONN[m] = np.stack(CONN[m])   # (n_subj, 61, 61)

    np.savez(CONN_CACHE, conn=CONN, subj_ids=np.array(SUBJ_IDS))
    log.info(f'Salvato: {CONN_CACHE}')

N_SUBJ = len(SUBJ_IDS)
log.info(f'Soggetti validi: {N_SUBJ}')
for m in METRICS:
    log.info(f'  {m}: {CONN[m].shape}')

## §3 — Clustering K-Means + Silhouette

Per ogni metrica: vettorizza triangolo superiore → PCA → K-means con k ∈ {2,3,4}.
Silhouette score per selezionare k ottimale.

In [ ]:
def upper_tri(mat):
    """(61,61) → vettore triangolo superiore senza diagonale (1830D)"""
    idx = np.triu_indices(mat.shape[0], k=1)
    return mat[idx]

def vectorize_conn(conn_3d):
    """(n_subj, 61, 61) → (n_subj, 1830)"""
    return np.stack([upper_tri(conn_3d[i]) for i in range(len(conn_3d))])

CLUSTER_RESULTS = {}   # metric → {'labels_k': {k: array}, 'sil': {k: float}, 'feat_pca': array}

fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 4), sharey=True)
fig.suptitle('Silhouette Score per Metrica e k', fontsize=13, fontweight='bold')

for ax, metric in zip(axes, METRICS):
    X_raw = vectorize_conn(CONN[metric])           # (n_subj, 1830)
    X_sc  = StandardScaler().fit_transform(X_raw)  # normalizza feature

    # PCA a 20 componenti per clustering stabile
    n_comp = min(20, X_sc.shape[0] - 1, X_sc.shape[1])
    pca    = PCA(n_components=n_comp, random_state=RANDOM_SEED)
    X_pca  = pca.fit_transform(X_sc)               # (n_subj, 20)
    var_exp = pca.explained_variance_ratio_.sum()

    sil_scores = {}
    labels_k   = {}
    for k in K_LIST:
        km = KMeans(n_clusters=k, n_init=N_INIT_KMEANS, random_state=RANDOM_SEED)
        labels = km.fit_predict(X_pca)
        sil    = silhouette_score(X_pca, labels) if len(set(labels)) > 1 else 0.0
        sil_scores[k] = sil
        labels_k[k]   = labels
        log.info(f'{metric}  k={k}  silhouette={sil:.4f}')

    CLUSTER_RESULTS[metric] = {
        'labels_k': labels_k,
        'sil':      sil_scores,
        'feat_pca': X_pca,
        'var_exp':  float(var_exp),
    }

    ax.bar([str(k) for k in K_LIST], [sil_scores[k] for k in K_LIST],
           color=['#2C7BB6', '#1A9641', '#D7191C'], alpha=0.85)
    ax.set_title(f'{metric}\n(PCA var={var_exp:.1%})', fontsize=10)
    ax.set_xlabel('k'); ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 0.6)

axes[0].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg16_silhouette.png', dpi=150)
plt.show()
log.info('Silhouette plot salvato')

## §4 — Grand Mean: Matrice Media Globale (tutti i soggetti)

`GRAND_MEAN[metric]` = media aritmetica delle N matrici soggetto → fingerprint medio del gruppo.

In [ ]:
import math

# ── k ottimale per metrica (argmax silhouette) ────────────────────────────────
BEST_K = {m: max(K_LIST, key=lambda k: CLUSTER_RESULTS[m]['sil'][k]) for m in METRICS}
log.info('k ottimale: ' + str({m: f'k={BEST_K[m]} sil={CLUSTER_RESULTS[m]["sil"][BEST_K[m]]:.3f}'
                                for m in METRICS}))

# ── Grand mean per metrica ────────────────────────────────────────────────────
GRAND_MEAN = {m: CONN[m].mean(axis=0) for m in METRICS}   # (61, 61) per metrica

# Colormap e range per metrica
CMAP_RANGE = {
    'pcc':     ('RdBu_r', -1.0, 1.0),
    'abs_pcc': ('hot',     0.0, 1.0),
    'im_pcc':  ('RdBu_r', -0.5, 0.5),
    'wpli':    ('YlOrRd',  0.0, 0.6),
    'plv':     ('YlOrRd',  0.0, 1.0),
}

fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 4))
fig.suptitle(f'Grand Mean — matrice media globale ({N_SUBJ} soggetti)',
             fontsize=13, fontweight='bold')

for ax, metric in zip(axes, METRICS):
    cmap, vmin, vmax = CMAP_RANGE[metric]
    im = ax.imshow(GRAND_MEAN[metric], cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation='nearest')
    ax.set_title(f'{metric}\n(best k={BEST_K[metric]})', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg16_grand_mean_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
log.info(f'Grand mean salvato per {len(METRICS)} metriche')

## §5 — Matrici Individuali per Cluster

Per ogni cluster k: griglia di matrici per-soggetto **raggruppate per cluster**.
Ogni cella = matrice media del soggetto (media su tutti i suoi trial).
Ultima colonna = media del cluster. Penultima = grand mean.

Metrica primaria: **abs_pcc** (topologia principale del progetto).

In [ ]:
PRIMARY_METRIC = 'abs_pcc'   # metrica principale per visualizzazione dettagliata
COLORS_CLUSTER = ['#2C7BB6', '#D7191C', '#1A9641', '#FF7F00']

cmap, vmin, vmax = CMAP_RANGE[PRIMARY_METRIC]

for metric in [PRIMARY_METRIC]:   # espandi a METRICS se vuoi tutte e 5
    k      = BEST_K[metric]
    labels = CLUSTER_RESULTS[metric]['labels_k'][k]
    conn   = CONN[metric]          # (n_subj, 61, 61)
    grand  = GRAND_MEAN[metric]    # (61, 61)
    sil    = CLUSTER_RESULTS[metric]['sil'][k]

    for c in range(k):
        mask          = labels == c
        subj_indices  = np.where(mask)[0]
        subj_ids_c    = [SUBJ_IDS[i] for i in subj_indices]
        mats_c        = conn[mask]           # (n_c, 61, 61)
        cluster_mean  = mats_c.mean(axis=0)  # (61, 61)

        n_subj_c  = len(subj_ids_c)
        n_panels  = n_subj_c + 2             # soggetti + cluster_mean + grand_mean
        ncols     = min(8, n_panels)
        nrows     = math.ceil(n_panels / ncols)

        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(2.8 * ncols, 2.8 * nrows + 0.8))
        fig.suptitle(
            f'{metric} | Cluster {c} — {n_subj_c} soggetti '
            f'(k={k}, sil={sil:.3f})',
            fontsize=11, fontweight='bold',
            color=COLORS_CLUSTER[c]
        )
        axes_flat = np.array(axes).flatten()

        # ── soggetti nel cluster ───────────────────────────────────────────
        for plot_i, (sid, mat) in enumerate(zip(subj_ids_c, mats_c)):
            ax = axes_flat[plot_i]
            ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax,
                      interpolation='nearest')
            ax.set_title(f'P{sid:03d}', fontsize=8,
                         color=COLORS_CLUSTER[c], fontweight='bold')
            ax.axis('off')

        # ── cluster mean ──────────────────────────────────────────────────
        ax_cm = axes_flat[n_subj_c]
        im_cm = ax_cm.imshow(cluster_mean, cmap=cmap, vmin=vmin, vmax=vmax,
                              interpolation='nearest')
        ax_cm.set_title(f'Cluster {c}\nmean', fontsize=8, fontweight='bold')
        ax_cm.axis('off')
        plt.colorbar(im_cm, ax=ax_cm, fraction=0.05, pad=0.02)

        # ── grand mean ────────────────────────────────────────────────────
        ax_gm = axes_flat[n_subj_c + 1]
        im_gm = ax_gm.imshow(grand, cmap=cmap, vmin=vmin, vmax=vmax,
                              interpolation='nearest')
        ax_gm.set_title('Grand\nmean', fontsize=8, fontweight='bold', color='gray')
        ax_gm.axis('off')
        plt.colorbar(im_gm, ax=ax_gm, fraction=0.05, pad=0.02)

        # ── nascondi pannelli vuoti ────────────────────────────────────────
        for i in range(n_panels, len(axes_flat)):
            axes_flat[i].axis('off')

        plt.tight_layout()
        fname = FIG_DIR / f'eeg16_{metric}_cluster{c}_k{k}_subjects.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        log.info(f'Cluster {c}: {n_subj_c} soggetti → {fname.name}')

## §7 — Dendrogram Gerarchico

Clustering gerarchico (Ward linkage) su abs_pcc: mostra la struttura naturale dei soggetti.

## §6 — Comparazione: Soggetto vs Grand Mean

`diff_i = CONN[metric][i] - GRAND_MEAN[metric]`

Ogni pannello mostra quanto il soggetto i si discosta dalla media globale.
- **Rosso** = connettività superiore alla media (over-connected)
- **Blu** = connettività inferiore alla media (under-connected)

Soggetti ordinati per cluster (colore del titolo). Utile per identificare pattern cluster-specifici.

In [ ]:
metric  = PRIMARY_METRIC
k       = BEST_K[metric]
labels  = CLUSTER_RESULTS[metric]['labels_k'][k]
grand   = GRAND_MEAN[metric]       # (61, 61)
conn    = CONN[metric]             # (n_subj, 61, 61)

# Ordina soggetti per cluster (C0 prima, C1 dopo, ...)
order       = np.argsort(labels)
sids_sorted = [SUBJ_IDS[i] for i in order]
labs_sorted = labels[order]

# Soglia diff colormap: 30% del range grand mean come riferimento
diff_abs = np.abs(conn - grand[None, :, :]).max()
d_clamp  = min(float(diff_abs) * 0.8, 0.4)

ncols = min(8, N_SUBJ + 1)
nrows = math.ceil((N_SUBJ + 1) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(2.8 * ncols, 2.8 * nrows + 0.8))
fig.suptitle(
    f'{metric} | Soggetto − Grand Mean (ordinati per cluster k={k})\n'
    f'colormap RdBu_r  ±{d_clamp:.2f}  |  rosso = over-connected, blu = under-connected',
    fontsize=10, fontweight='bold'
)
axes_flat = np.array(axes).flatten()

# ── Grand mean (pannello 0) ───────────────────────────────────────────────────
cmap_gm, vmin_gm, vmax_gm = CMAP_RANGE[metric]
im_gm = axes_flat[0].imshow(grand, cmap=cmap_gm, vmin=vmin_gm, vmax=vmax_gm,
                              interpolation='nearest')
axes_flat[0].set_title('Grand mean', fontsize=8, fontweight='bold', color='gray')
axes_flat[0].axis('off')
plt.colorbar(im_gm, ax=axes_flat[0], fraction=0.05, pad=0.02)

# ── Differenze soggetto per soggetto ─────────────────────────────────────────
for plot_i, (subj_idx, sid, cluster_c) in enumerate(
        zip(order, sids_sorted, labs_sorted), start=1):
    diff = conn[subj_idx] - grand       # (61, 61)
    ax   = axes_flat[plot_i]
    ax.imshow(diff, cmap='RdBu_r', vmin=-d_clamp, vmax=d_clamp,
              interpolation='nearest')
    ax.set_title(f'P{sid:03d} [C{cluster_c}]', fontsize=7,
                 color=COLORS_CLUSTER[cluster_c], fontweight='bold')
    ax.axis('off')

# ── nascondi pannelli vuoti ────────────────────────────────────────────────────
for i in range(N_SUBJ + 1, len(axes_flat)):
    axes_flat[i].axis('off')

plt.tight_layout()
fname = FIG_DIR / f'eeg16_{metric}_diff_vs_grandmean_k{k}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
log.info(f'Comparison salvato: {fname.name}')

# ── Stampa deviazione media per cluster ───────────────────────────────────────
print(f'\n── Deviazione media |soggetto - grand mean| per cluster ({metric}) ──')
for c in range(k):
    idxs_c  = np.where(labels == c)[0]
    diffs_c = np.abs(conn[idxs_c] - grand[None]).mean(axis=(1, 2))
    print(f'  Cluster {c} (n={len(idxs_c)}): '
          f'mean_dev={diffs_c.mean():.4f} ± {diffs_c.std():.4f}  '
          f'[{", ".join(f"P{SUBJ_IDS[i]:03d}={diffs_c[j]:.3f}" for j,i in enumerate(idxs_c))}]')

In [ ]:
for metric in ['abs_pcc', 'wpli']:   # le due metriche principali
    X_raw = vectorize_conn(CONN[metric])
    X_sc  = StandardScaler().fit_transform(X_raw)

    Z = linkage(X_sc, method='ward')

    fig, ax = plt.subplots(figsize=(max(12, N_SUBJ * 0.3), 5))
    dendrogram(Z,
               labels=[f'P{sid:03d}' for sid in SUBJ_IDS],
               ax=ax, leaf_rotation=90, leaf_font_size=7,
               color_threshold=0.7 * max(Z[:, 2]))
    ax.set_title(f'Clustering gerarchico soggetti — {metric} (Ward)', fontsize=12)
    ax.set_ylabel('Distanza Ward')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'eeg16_dendrogram_{metric}.png', dpi=150)
    plt.show()

## §8 — Post-hoc: Overlay Accuracy

I cluster EEG riflettono la capacità di imagined speech?
Carica risultati EEG_12 (W-HGNN subject-specific) e overlay sui cluster.

In [ ]:
# Carica accuracy disponibili
acc_data = {}
for label, fpath in [('EEG_12', ACC_FILE_12), ('EEG_13', ACC_FILE_13)]:
    if fpath.exists():
        df = pd.read_csv(fpath)
        if 'Subject' in df.columns:
            df['sid'] = df['Subject'].str.extract(r'(\d+)').astype(int)
        if 'Test bAcc' in df.columns:
            acc_data[label] = dict(zip(df['sid'], df['Test bAcc']))
        elif 'test_bacc' in df.columns:
            acc_data[label] = dict(zip(df['sid'], df['test_bacc']))
        log.info(f'Caricato {label}: {len(acc_data[label])} soggetti')
    else:
        log.warning(f'{label} non trovato: {fpath}')

if not acc_data:
    print('[INFO] Nessun file accuracy trovato — esegui EEG_12/13 prima.')
else:
    chance = 1 / N_CLASSES

    for acc_label, acc_dict in acc_data.items():
        for metric in ['abs_pcc', 'wpli']:
            k      = BEST_K[metric]
            labels = CLUSTER_RESULTS[metric]['labels_k'][k]
            X_pca  = CLUSTER_RESULTS[metric]['feat_pca'][:, :2]

            accs = np.array([acc_dict.get(sid, np.nan) for sid in SUBJ_IDS])

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            fig.suptitle(f'Post-hoc: {acc_label} accuracy × cluster {metric} (k={k})',
                         fontsize=12, fontweight='bold')

            # ── Scatter: colore=accuracy, bordo=cluster ────────────────────
            ax = axes[0]
            valid = ~np.isnan(accs)
            sc = ax.scatter(X_pca[valid, 0], X_pca[valid, 1],
                            c=accs[valid], cmap='RdYlGn',
                            s=120, vmin=chance, vmax=accs[valid].max(),
                            edgecolors=[COLORS_CLUSTER[labels[i]] for i in np.where(valid)[0]],
                            linewidths=2.5)
            plt.colorbar(sc, ax=ax, label='bAcc')
            ax.axhline(0, color='k', alpha=0.2); ax.axvline(0, color='k', alpha=0.2)
            ax.set_title('Scatter: fill=accuracy, bordo=cluster'); ax.grid(alpha=0.2)
            for i in np.where(valid)[0]:
                ax.annotate(f'P{SUBJ_IDS[i]:03d}', (X_pca[i, 0], X_pca[i, 1]),
                            fontsize=5, xytext=(2, 2), textcoords='offset points')

            # ── Box plot accuracy per cluster ──────────────────────────────
            ax2 = axes[1]
            cluster_accs = [accs[labels == c] for c in range(k)]
            cluster_accs_valid = [a[~np.isnan(a)] for a in cluster_accs]
            bp = ax2.boxplot(cluster_accs_valid,
                             patch_artist=True, notch=False,
                             medianprops=dict(color='black', linewidth=2))
            for patch, c in zip(bp['boxes'], range(k)):
                patch.set_facecolor(COLORS_CLUSTER[c]); patch.set_alpha(0.7)
            ax2.axhline(chance, color='k', linestyle='--', linewidth=1.5,
                        label=f'Chance ({chance:.0%})')
            ax2.set_xticklabels([f'Cluster {c}\n(n={len(cluster_accs_valid[c])})'
                                  for c in range(k)])
            ax2.set_ylabel('Balanced Accuracy'); ax2.legend()
            ax2.set_title('Distribuzione accuracy per cluster'); ax2.grid(axis='y', alpha=0.3)

            for c in range(k):
                cv = cluster_accs_valid[c]
                if len(cv) > 0:
                    print(f'{metric} k={k} Cluster {c}: mean={cv.mean():.4f} ± {cv.std():.4f}  (n={len(cv)})')

            plt.tight_layout()
            fname = FIG_DIR / f'eeg16_posthoc_{acc_label}_{metric}_k{k}.png'
            plt.savefig(fname, dpi=150)
            plt.show()
            log.info(f'Salvato: {fname}')

## §9 — Riepilogo Cluster

Tabella: soggetto → cluster per ogni metrica (k ottimale).

In [ ]:
rows = []
for i, sid in enumerate(SUBJ_IDS):
    row = {'Subject': f'P{sid:03d}'}
    for m in METRICS:
        k = BEST_K[m]
        row[f'cluster_{m}'] = CLUSTER_RESULTS[m]['labels_k'][k][i]
    # aggiungi accuracy se disponibile
    for acc_label, acc_dict in acc_data.items():
        row[f'acc_{acc_label}'] = round(acc_dict.get(sid, np.nan), 4)
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary.to_csv(FIG_DIR / 'eeg16_subject_clusters.csv', index=False)
print(df_summary.to_string(index=False))

# Concordanza tra metriche
print('\n── k ottimali ──')
for m in METRICS:
    k   = BEST_K[m]
    sil = CLUSTER_RESULTS[m]['sil'][k]
    print(f'  {m}: k={k}  silhouette={sil:.4f}')

## §10 — Raw Trial: P022 (outlier) vs P068 (top performer)

Confronto visivo: connectivity matrix media, segnale EEG grezzo (canali selezionati), distribuzione per-canale.

In [ ]:
COMPARE_SUBJS = {
    'P022 (outlier)':    22,
    'P068 (top EEG_13)': 68,
}
N_TRIAL_SHOW = 5   # quanti trial sovrapporre per traccia EEG
# Canali rappresentativi: frontale, centrale, occipitale, temporale
SHOW_CH = [0, 10, 20, 30, 40, 50, 60]

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, len(COMPARE_SUBJS), figure=fig,
                         hspace=0.45, wspace=0.35)
fig.suptitle('P022 vs P068 — raw trial comparison (abs_pcc)', fontsize=13, fontweight='bold')

cmap_c, vmin_c, vmax_c = CMAP_RANGE['abs_pcc']

for col, (label, sid) in enumerate(COMPARE_SUBJS.items()):
    # ── Carica trial da hypergraphs_pruned_abs_pcc ────────────────────────
    idx_s = build_index('abs_pcc')
    paths_s = idx_s.get(sid, [])
    if not paths_s:
        print(f'[WARN] Nessun trial trovato per P{sid:03d}')
        continue

    trials_x   = []
    trials_adj = []
    for p in paths_s[:50]:   # prime 50 trial
        try:
            d = torch.load(p, weights_only=False)
            trials_x.append(d['x'].float().numpy())       # (61, 384)
            trials_adj.append(d['adj'].float().numpy())   # (61, 61)
        except Exception:
            continue

    mean_adj = np.mean(trials_adj, axis=0)   # (61, 61)
    sid_idx  = SUBJ_IDS.index(sid) if sid in SUBJ_IDS else None
    acc12    = acc_data.get('EEG_12', {}).get(sid, np.nan)
    acc13    = acc_data.get('EEG_13', {}).get(sid, np.nan)

    # ── Row 0: connectivity matrix media ─────────────────────────────────
    ax0 = fig.add_subplot(gs[0, col])
    im  = ax0.imshow(mean_adj, cmap=cmap_c, vmin=vmin_c, vmax=vmax_c,
                     interpolation='nearest')
    ax0.set_title(f'{label}\nacc12={acc12:.3f}  acc13={acc13:.3f}', fontsize=10)
    ax0.axis('off')
    plt.colorbar(im, ax=ax0, fraction=0.046, pad=0.02)

    # ── Row 1: EEG raw traces (N_TRIAL_SHOW trial sovrapposti) ────────────
    ax1 = fig.add_subplot(gs[1, col])
    t   = np.arange(N_SAMPLES) / 256.0   # asse temporale in secondi
    colors_t = plt.cm.tab10(np.linspace(0, 1, N_TRIAL_SHOW))
    for ti, x in enumerate(trials_x[:N_TRIAL_SHOW]):
        # Normalizza per visualizzazione: z-score per canale
        x_z = (x - x.mean(axis=1, keepdims=True)) / (x.std(axis=1, keepdims=True) + 1e-8)
        offset = 0
        for ch in SHOW_CH:
            ax1.plot(t, x_z[ch] + offset, color=colors_t[ti], alpha=0.5, linewidth=0.6)
            offset += 4
    ax1.set_xlabel('Tempo (s)'); ax1.set_ylabel('Canali (offset)')
    ax1.set_title(f'{N_TRIAL_SHOW} trial sovrapposti — {len(SHOW_CH)} canali', fontsize=9)
    ax1.set_yticks(np.arange(len(SHOW_CH)) * 4)
    ax1.set_yticklabels([f'ch{c}' for c in SHOW_CH], fontsize=7)
    ax1.grid(axis='x', alpha=0.3)

    # ── Row 2: distribuzione ampiezza RMS per canale ──────────────────────
    ax2 = fig.add_subplot(gs[2, col])
    rms_per_ch = np.array([np.sqrt(np.mean(x**2, axis=1)) for x in trials_x[:30]])
    # (n_trials, 61) → media e std per canale
    ax2.errorbar(np.arange(N_CHANNELS), rms_per_ch.mean(axis=0),
                 yerr=rms_per_ch.std(axis=0),
                 fmt='-o', markersize=2, linewidth=0.8, capsize=2, color='steelblue')
    ax2.set_xlabel('Canale'); ax2.set_ylabel('RMS (µV)')
    ax2.set_title('RMS per canale (media ± std su 30 trial)', fontsize=9)
    ax2.grid(alpha=0.3)

plt.savefig(FIG_DIR / 'eeg16_p022_vs_p068_raw.png', dpi=150, bbox_inches='tight')
plt.show()
log.info('Confronto P022 vs P068 salvato')

## §11 — k=2 Forzato: i due mega-cluster si allineano con l'accuracy?

Forza k=2 su ogni metrica (ignorando il silhouette ottimale) e verifica se la bipartizione
riflette un'effettiva differenza di performance IS.

In [ ]:
from scipy.stats import mannwhitneyu

K_FORCED = 2

if not acc_data:
    print('[INFO] Nessun file accuracy — esegui §8 prima.')
else:
    chance = 1 / N_CLASSES

    # ── Tabella riassuntiva: ogni metrica × cluster × accuracy ────────────
    summary_rows = []

    fig, axes = plt.subplots(len(METRICS), len(acc_data),
                              figsize=(7 * len(acc_data), 3.5 * len(METRICS)),
                              squeeze=False)
    fig.suptitle(f'k=2 forzato — accuracy per cluster (tutte le metriche)',
                 fontsize=13, fontweight='bold')

    for row_i, metric in enumerate(METRICS):
        # Ricalcola k=2 se non già in CLUSTER_RESULTS (dovrebbe esserci da §3)
        if K_FORCED in CLUSTER_RESULTS[metric]['labels_k']:
            labels2 = CLUSTER_RESULTS[metric]['labels_k'][K_FORCED]
        else:
            X_raw = vectorize_conn(CONN[metric])
            X_sc  = StandardScaler().fit_transform(X_raw)
            n_comp = min(20, X_sc.shape[0] - 1, X_sc.shape[1])
            X_pca2 = PCA(n_components=n_comp, random_state=RANDOM_SEED).fit_transform(X_sc)
            labels2 = KMeans(n_clusters=K_FORCED, n_init=N_INIT_KMEANS,
                              random_state=RANDOM_SEED).fit_predict(X_pca2)

        for col_i, (acc_label, acc_dict) in enumerate(acc_data.items()):
            accs = np.array([acc_dict.get(sid, np.nan) for sid in SUBJ_IDS])

            c0 = accs[labels2 == 0]; c0 = c0[~np.isnan(c0)]
            c1 = accs[labels2 == 1]; c1 = c1[~np.isnan(c1)]

            # Mann-Whitney U test (non parametrico, piccolo n)
            if len(c0) > 1 and len(c1) > 1:
                stat, pval = mannwhitneyu(c0, c1, alternative='two-sided')
            else:
                pval = np.nan

            summary_rows.append({
                'metric': metric, 'acc_source': acc_label,
                'C0_n': len(c0), 'C0_mean': c0.mean() if len(c0) else np.nan,
                'C0_std': c0.std() if len(c0) else np.nan,
                'C1_n': len(c1), 'C1_mean': c1.mean() if len(c1) else np.nan,
                'C1_std': c1.std() if len(c1) else np.nan,
                'delta': abs(c0.mean() - c1.mean()) if (len(c0) and len(c1)) else np.nan,
                'p_mwu': pval,
            })

            ax = axes[row_i][col_i]
            bp = ax.boxplot([c0, c1], patch_artist=True, notch=False,
                            medianprops=dict(color='black', linewidth=2))
            for patch, c in zip(bp['boxes'], range(2)):
                patch.set_facecolor(COLORS_CLUSTER[c]); patch.set_alpha(0.75)
            ax.axhline(chance, color='k', linestyle='--', linewidth=1.2,
                       label=f'Chance ({chance:.0%})')
            ax.set_xticklabels([f'C0\n(n={len(c0)})', f'C1\n(n={len(c1)})'])
            ax.set_title(f'{metric} | {acc_label}  p={pval:.3f}', fontsize=9)
            ax.set_ylabel('bAcc'); ax.grid(axis='y', alpha=0.3)
            # Annota delta
            delta = abs(c0.mean() - c1.mean()) if (len(c0) and len(c1)) else 0
            ax.text(0.5, 0.02, f'Δ={delta:.4f}', transform=ax.transAxes,
                    ha='center', fontsize=8, color='gray')

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eeg16_k2_forced_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Tabella riassuntiva stampata ───────────────────────────────────────
    df_k2 = pd.DataFrame(summary_rows)
    df_k2['sig'] = df_k2['p_mwu'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    )
    print('\n── k=2 forzato: test Mann-Whitney U ──')
    print(df_k2[['metric','acc_source','C0_mean','C0_std','C1_mean','C1_std',
                  'delta','p_mwu','sig']].to_string(index=False, float_format='{:.4f}'.format))

    best = df_k2.sort_values('delta', ascending=False).iloc[0]
    print(f'\nMigliore separazione: {best["metric"]} × {best["acc_source"]}  '
          f'Δ={best["delta"]:.4f}  p={best["p_mwu"]:.4f}  ({best["sig"]})')